# Parcial para la casa · Raíces A — la ecuación de Kepler con datos de la NASA

**Métodos Numéricos · UNAL Manizales · 2026-2**

En 1609 Kepler necesitaba resolver, para cada fecha, la ecuación

$$M = E - e\,\sin E$$

(M: anomalía media, e: excentricidad, E: anomalía excéntrica). No se puede despejar E: hay que **buscar una raíz**. En este cuaderno resuelves Kepler con bisección y con Newton usando elementos orbitales reales del JPL (NASA), calculas la anomalía verdadera ν y la **comparas con la que publica la NASA**.

**Fuente de los datos.** JPL Horizons System, Solar System Dynamics Group, NASA/JPL-Caltech. API pública sin clave: `https://ssd.jpl.nasa.gov/api/horizons.api` con `EPHEM_TYPE='ELEMENTS'` y `CENTER='500@10'` (centro: el Sol). Columnas usadas: `EC` = e, `MA` = M (grados), `TA` = ν (grados).
**Fecha de consulta de la copia congelada:** 15 de octubre de 2026.
**Licencia / uso:** datos públicos del JPL/NASA, de uso libre con atribución (cita "JPL Horizons, consultado el 15/10/2026"); condiciones en ssd.jpl.nasa.gov.

Reglas del cuaderno: solo cambias lo marcado `# <-- CAMBIA ESTO`. Antes de cada experimento hay una caja **Predice en tu guía antes de correr**: escribe tu predicción en papel *antes* de ejecutar.

## Celda 1 · Carga de datos reales (API de la NASA + respaldo congelado)

La celda intenta consultar la API de Horizons en vivo. Si no hay internet o la API falla, usa la copia congelada (los mismos números, copiados textualmente el 15/10/2026). Al final te dice de dónde salió cada dato.

In [ ]:
import re, urllib.parse, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL_API = "https://ssd.jpl.nasa.gov/api/horizons.api"

# (nombre, comando de Horizons, fecha de inicio, fecha de fin)
CUERPOS = [
    ("Marte",                    "499",              "2026-10-15", "2026-10-16"),
    ("Mercurio",                 "199",              "2026-10-15", "2026-10-16"),
    ("Halley (2026)",            "90000030",         "2026-10-15", "2026-10-16"),
    ("Halley (1 mar 1986)",      "90000030",         "1986-03-01", "1986-03-02"),
    ("Hale-Bopp (2026)",         "DES=C/1995 O1;",   "2026-10-15", "2026-10-16"),
]

# Copia congelada: primera fila del bloque $$SOE ... $$EOE de cada consulta (15/10/2026)
RESPALDO = {
    "Marte":               dict(EC=9.341138744261415E-02, MA=1.062035241578232E+02, TA=1.161083937824729E+02),
    "Mercurio":            dict(EC=2.056386626013959E-01, MA=2.521510986901104E+02, TA=2.317942556942285E+02),
    "Halley (2026)":       dict(EC=9.680308194898556E-01, MA=1.940065702167612E+02, TA=1.809093929781099E+02),
    "Halley (1 mar 1986)": dict(EC=9.672757899254457E-01, MA=2.534454496239656E-01, TA=5.193163014067228E+01),
    "Hale-Bopp (2026)":    dict(EC=9.948892134295946E-01, MA=4.361806722999568E+00, TA=1.656675636899563E+02),
}

def consultar_horizons(comando, inicio, fin):
    """Consulta la API y devuelve EC, MA, TA de la primera fila de elementos."""
    params = {
        "format": "text", "COMMAND": f"'{comando}'", "OBJ_DATA": "'NO'",
        "MAKE_EPHEM": "'YES'", "EPHEM_TYPE": "'ELEMENTS'", "CENTER": "'500@10'",
        "START_TIME": f"'{inicio}'", "STOP_TIME": f"'{fin}'", "STEP_SIZE": "'1 d'",
    }
    url = URL_API + "?" + urllib.parse.urlencode(params, quote_via=urllib.parse.quote)
    with urllib.request.urlopen(url, timeout=20) as r:
        texto = r.read().decode("utf-8")
    bloque = texto.split("$$SOE")[1].split("$$EOE")[0]
    leer = lambda clave: float(re.search(clave + r"\s*=\s*([-+0-9.Ee]+)", bloque).group(1))
    return dict(EC=leer("EC"), MA=leer("MA"), TA=leer("TA"))

filas = []
for nombre, comando, inicio, fin in CUERPOS:
    try:
        d = consultar_horizons(comando, inicio, fin)
        origen = "API en vivo"
    except Exception as err:
        d = RESPALDO[nombre]
        origen = "respaldo congelado"
    filas.append(dict(cuerpo=nombre, fecha=inicio, e=d["EC"], M_grados=d["MA"], TA_nasa_grados=d["TA"], origen=origen))

datos = pd.DataFrame(filas).set_index("cuerpo")
pd.set_option("display.float_format", lambda x: f"{x:.10g}")
datos

## Celda 2 · Las herramientas: f, f′, bisección y Newton

f(E) = E − e·sen E − M y f′(E) = 1 − e·cos E. Ambos métodos devuelven la **historia** de iterados para que puedas mirar los errores.

**Para completar (opcional, como en los labs):** escribe el paso de Newton en `paso_newton`. Si lo dejas en `None`, la celda 2b usa la versión de respaldo.

In [ ]:
def f(E, e, M):
    return E - e*np.sin(E) - M

def fprima(E, e, M):
    return 1 - e*np.cos(E)

def paso_newton(E, e, M):
    # COMPLETA: devuelve el siguiente iterado de Newton a partir de E
    return None  # <-- CAMBIA ESTO

def biseccion(e, M, a=0.0, b=2*np.pi, tol=1e-10):
    """Bisección en [a, b]. Para cuando la cota (b-a)/2^n es menor que tol.
    Devuelve (raíz, número de iteraciones, historia de puntos medios)."""
    a0, b0 = a, b
    fa = f(a, e, M)
    historia = []
    n = 0
    while True:
        n += 1
        p = a + (b - a)/2
        historia.append(p)
        fp = f(p, e, M)
        if fa*fp > 0:
            a, fa = p, fp
        else:
            b = p
        if (b0 - a0)/2**n < tol:
            return p, n, historia

def newton(e, M, E0, tol=1e-10, maxit=50):
    """Newton desde E0. Para cuando |E_{k+1} - E_k| < tol.
    Devuelve (raíz, iteraciones o None si no convergió, historia)."""
    E = E0
    historia = [E0]
    for k in range(1, maxit + 1):
        E_nuevo = paso_newton(E, e, M)
        historia.append(E_nuevo)
        if abs(E_nuevo - E) < tol:
            return E_nuevo, k, historia
        E = E_nuevo
    return E, None, historia

def anomalia_verdadera(E, e):
    """nu a partir de E: tan(nu/2) = sqrt((1+e)/(1-e)) tan(E/2).
    Se usa arctan2 (misma fórmula) para que nu quede en el cuadrante correcto. Devuelve grados en [0, 360)."""
    nu = 2*np.arctan2(np.sqrt(1 + e)*np.sin(E/2), np.sqrt(1 - e)*np.cos(E/2))
    return np.degrees(nu) % 360

### Celda 2b · Respaldo del paso de Newton (no la modifiques)

In [ ]:
if paso_newton(1.0, 0.5, 0.5) is None:
    def paso_newton(E, e, M):
        return E - f(E, e, M)/fprima(E, e, M)
    print("Usando el paso de Newton de respaldo.")
else:
    prueba = paso_newton(1.0, 0.5, 0.5)
    print("Tu paso de Newton desde E=1, e=0,5, M=0,5 da", prueba,
          "(el de respaldo da", 1.0 - f(1.0, 0.5, 0.5)/fprima(1.0, 0.5, 0.5), ")")

## Celda 3 · Ítem II.1 — ¿Cuántas iteraciones? Bisección vs Newton con Marte

**Predice en tu guía antes de correr:**
(a) ¿Cuántas iteraciones necesita la bisección en [0, 2π] para garantizar error < 10⁻¹⁰? (Usa la cota de la Parte I.)
(b) ¿Cuántas necesitará Newton desde E₀ = M para Marte (e ≈ 0,09)? ¿Más o menos que la bisección?

In [ ]:
cuerpo = "Marte"   # <-- CAMBIA ESTO: "Mercurio", "Halley (2026)", "Halley (1 mar 1986)" o "Hale-Bopp (2026)"
tol = 1e-10        # <-- CAMBIA ESTO

e = datos.loc[cuerpo, "e"]
M = np.radians(datos.loc[cuerpo, "M_grados"])

E_bis, n_bis, _ = biseccion(e, M, tol=tol)
E_new, k_new, _ = newton(e, M, E0=M, tol=tol)

print(f"{cuerpo}: e = {e:.6f},  M = {M:.6f} rad")
print(f"  Bisección: {n_bis} iteraciones -> E = {E_bis:.12f}")
print(f"  Newton (E0 = M): {k_new} iteraciones -> E = {E_new:.12f}")
print(f"  Diferencia entre las dos raíces: {abs(E_bis - E_new):.2e}")

## Celda 4 · Ítem II.2 — Tu ν contra la ν de la NASA

**Predice en tu guía antes de correr:** si resuelves Kepler con tolerancia 10⁻¹⁰ y calculas ν, ¿de qué tamaño esperas la diferencia con la `TA` de la NASA (en grados)? ¿10⁻¹? ¿10⁻⁶? ¿10⁻¹²?

In [ ]:
filas = []
for cuerpo in datos.index:
    e = datos.loc[cuerpo, "e"]
    M = np.radians(datos.loc[cuerpo, "M_grados"])
    E, _, _ = biseccion(e, M, tol=1e-12)
    nu = anomalia_verdadera(E, e)
    ta = datos.loc[cuerpo, "TA_nasa_grados"]
    filas.append(dict(cuerpo=cuerpo, e=e, M_grados=datos.loc[cuerpo, "M_grados"], E_rad=E,
                      nu_tuya_grados=nu, TA_nasa_grados=ta, diferencia_grados=abs(nu - ta)))
comparacion = pd.DataFrame(filas).set_index("cuerpo")
comparacion

## Celda 5 · Ítem II.3 — El cometa Hale-Bopp: cuando Newton se porta mal

Hale-Bopp tiene e ≈ 0,9949 y hoy su M es apenas ≈ 4,36° ≈ 0,076 rad.

**Predice en tu guía antes de correr:**
(a) Calcula a mano f′(E₀) = 1 − e·cos E₀ con E₀ = M = 0,076. ¿Qué tan grande será el primer paso de Newton, −f(E₀)/f′(E₀)?
(b) ¿Cuántas iteraciones necesitará Newton desde E₀ = M? ¿Y la bisección?

In [ ]:
cuerpo = "Hale-Bopp (2026)"
E0_opcion = "M"     # <-- CAMBIA ESTO: "M", "pi" o "M+e·senM"

e = datos.loc[cuerpo, "e"]
M = np.radians(datos.loc[cuerpo, "M_grados"])
E0 = {"M": M, "pi": np.pi, "M+e·senM": M + e*np.sin(M)}[E0_opcion]

E_new, k_new, hist = newton(e, M, E0)
E_bis, n_bis, _ = biseccion(e, M)

tabla = pd.DataFrame({"E_k": hist, "f'(E_k)": [fprima(x, e, M) for x in hist]})
tabla.index.name = "k"
print(f"{cuerpo}: e = {e:.6f}, M = {M:.6f} rad, E0 = {E0_opcion} = {E0:.6f}")
print(f"Newton: {k_new} iteraciones.  Bisección: {n_bis} iteraciones.")
print(f"Raíz de Newton {E_new:.10f}  vs  raíz de bisección {E_bis:.10f}")
tabla

### Celda 5b · Barrido: Newton desde E₀ = M para M pequeños (e de Hale-Bopp)

**Predice en tu guía antes de correr:** para e = 0,9949, ¿para qué M de esta lista crees que Newton desde E₀ = M falla (no converge en 50 iteraciones)? ¿La bisección falla alguna vez?

In [ ]:
e = datos.loc["Hale-Bopp (2026)", "e"]
M_lista = [0.001, 0.01, 0.035, 0.0761, 0.2, 1.0]   # <-- CAMBIA ESTO

filas = []
for M in M_lista:
    E_n, k, _ = newton(e, M, E0=M)
    E_b, n, _ = biseccion(e, M)
    ok = (k is not None) and abs(E_n - E_b) < 1e-8
    filas.append(dict(M=M, fprima_en_E0=fprima(M, e, M),
                      newton_iter=k if ok else "no convergió",
                      biseccion_iter=n, raiz=E_b))
pd.DataFrame(filas)

In [ ]:
# Mapa: iteraciones de Newton (E0 = M) para 500 valores de M en (0, 0,5]
Ms = np.linspace(0.001, 0.5, 500)
iters, fallos = [], []
for M in Ms:
    E_n, k, _ = newton(e, M, E0=M)
    E_b, _, _ = biseccion(e, M)
    if k is None or abs(E_n - E_b) > 1e-8:
        fallos.append(M)
    else:
        iters.append((M, k))

fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.scatter([m for m, _ in iters], [k for _, k in iters], s=12, color="#2a78d6", label="Newton converge")
ax.scatter(fallos, [50]*len(fallos), s=22, marker="x", color="#eb6834", label="no converge en 50 iter.")
ax.axhline(36, color="#888888", lw=1, ls="--")
ax.text(0.003, 36.8, "bisección: siempre 36", ha="left", va="bottom", fontsize=9, color="#555555")
ax.set_xlabel("M (rad)"); ax.set_ylabel("iteraciones")
ax.set_title(f"Newton desde E0 = M, e = {e:.4f}")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
plt.tight_layout(); plt.show()
print(f"Fallos de Newton: {len(fallos)} de {len(Ms)} valores de M")

## Celda 6 · Ítem II.4 — Leer el orden de convergencia en una tabla de errores

Usamos Marte. eₖ = |Eₖ − E\*|, donde E\* es la raíz calculada con mucha precisión.
Si el método es **lineal**, eₖ₊₁/eₖ se estabiliza en una constante < 1. Si es **cuadrático**, lo que se estabiliza es eₖ₊₁/eₖ².
Para la bisección también mostramos la **cota** (b − a)/2ⁿ⁺¹ de la Parte I.

**Predice en tu guía antes de correr:** ¿en cuál columna se ve una constante para la bisección? ¿Y para Newton? ¿Cuántas cifras correctas gana Newton por iteración al final?

In [ ]:
cuerpo = "Marte"   # <-- CAMBIA ESTO
e = datos.loc[cuerpo, "e"]
M = np.radians(datos.loc[cuerpo, "M_grados"])
E_ref, _, _ = newton(e, M, E0=M, tol=1e-15)

def tabla_errores(historia, n_filas, con_cota=False):
    err = [abs(x - E_ref) for x in historia[:n_filas]]
    filas = []
    for k in range(len(err)):
        ok = k > 0 and err[k-1] > 0 and err[k] > 0
        fila = dict(k=k, error=err[k],
                    razon_lineal=err[k]/err[k-1] if ok else np.nan,
                    razon_cuadratica=err[k]/err[k-1]**2 if ok else np.nan)
        if con_cota:
            fila["cota_(b-a)/2^(k+1)"] = 2*np.pi/2**(k+1)
        filas.append(fila)
    return pd.DataFrame(filas).set_index("k")

_, _, hb = biseccion(e, M, tol=1e-10)
_, _, hn = newton(e, M, E0=M, tol=1e-10)
print("BISECCIÓN (primeras 10 iteraciones; k = 0 es el primer punto medio)")
display(tabla_errores(hb, 10, con_cota=True))
n = 30
print(f"Bisección: razón promedio (geométrica) en {n} pasos = (e_{n}/e_0)^(1/{n}) =",
      (abs(hb[n] - E_ref)/abs(hb[0] - E_ref))**(1/n))
print()
print("NEWTON (k = 0 es E0 = M; un error 0 significa que ya se llegó a la precisión de la máquina)")
display(tabla_errores(hn, len(hn)))
print("Constante teórica de Newton |f''(E*)| / (2 |f'(E*)|) =",
      abs(e*np.sin(E_ref))/(2*abs(fprima(E_ref, e, M))))

## Celda 7 · Laboratorio del auditor (para la Parte III)

Herramienta libre para verificar lo que "reporta el agente" de la Parte III. Cambia e, el punto inicial y la malla de M, y mira el máximo de iteraciones y los fallos. **No basta con correrla:** en la guía tienes que justificar con la teoría.

**Predice en tu guía antes de correr:** para cada afirmación de la tabla de la Parte III que vayas a auditar, escribe primero si crees que es V o F y por qué; luego córrela con los parámetros que la ponen a prueba.

In [ ]:
e = 0.99              # <-- CAMBIA ESTO
E0_opcion = "pi"      # <-- CAMBIA ESTO: "M", "pi" o "M+e·senM"
n_puntos = 1000       # <-- CAMBIA ESTO
tol = 1e-10           # <-- CAMBIA ESTO
M_min = 0.001         # <-- CAMBIA ESTO (la malla va de M_min a 2*pi - M_min)

Ms = np.linspace(M_min, 2*np.pi - M_min, n_puntos)
its, fallos = [], []
for M in Ms:
    E0 = {"M": M, "pi": np.pi, "M+e·senM": M + e*np.sin(M)}[E0_opcion]
    E_n, k, _ = newton(e, M, E0, tol=tol)
    E_b, n_bis, _ = biseccion(e, M, tol=tol)
    if k is None or abs(E_n - E_b) > 1e-8:
        fallos.append(M)
    else:
        its.append(k)
print(f"e = {e}, E0 = {E0_opcion}, tol = {tol:g}, {n_puntos} valores de M en [{M_min:g}, 2pi - {M_min:g}]")
print(f"  Newton: máximo {max(its)} iteraciones, promedio {np.mean(its):.2f}, fallos: {len(fallos)}")
if fallos:
    print(f"  M donde falló: entre {min(fallos):.3f} y {max(fallos):.3f}")
print(f"  Bisección: {n_bis} iteraciones (para el último M)")